## Setup

Keep [`example_data.py`](example_data.py), [`data_fidelity_metrics.py`](data_fidelity_metrics.py), and [`missingness_fidelity_metrics.py`](missingness_fidelity_metrics.py) beside this notebook. The project environment supplies NumPy, pandas, Matplotlib, and IPython.

In [ ]:
#| label: setup

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from data_fidelity_metrics import hellinger_distance
from example_data import make_example_views
from missingness_fidelity_metrics import (
    marginal_missingness_details,
    missingness_hellinger_distance,
)

## Learning goals

By the end of this lesson, you should be able to:

- construct the missingness indicator matrices $M^X$ and $M^Y$;
- calculate the missing-value rate of each variable in $X$ and $Y$;
- convert each rate into an observed/missing probability distribution;
- calculate and average the marginal Hellinger distances; and
- explain what small and large metric values mean.

## 1. Metric, inputs, and notation

The **Marginal Missingness Hellinger Distance** is a missingness-fidelity metric. We write it as $\mathrm{Hel}_{\mathrm{miss}}^{\mathrm{marg}}$. It compares how often each variable is missing in two datasets. It examines one variable at a time. The word **marginal** means that it does not examine which variables are missing together in the same record.

We use the notation from the series:

- $X$ is the reference dataset, with dimensions $n_X\times d$.
- $Y$ is the comparison dataset, with dimensions $n_Y\times d$.
- $n_X$ and $n_Y$ are the numbers of records in $X$ and $Y$, and $d$ is the number of variables shared by both datasets.
- $X_j=\{x_{ij}\}_{i=1}^{n_X}$ and $Y_j=\{y_{ij}\}_{i=1}^{n_Y}$ are variable $j$ in the two datasets.
- $M^X$ and $M^Y$ are the missingness indicator matrices of $X$ and $Y$, with dimensions $n_X\times d$ and $n_Y\times d$.

This metric uses the original incomplete datasets $X$ and $Y$. It does not use the complete preprocessed datasets $\widetilde X$ and $\widetilde Y$.

In the shared example, $d=4$, $n_X=6$, and $n_Y=5$.

In [ ]:
#| label: original-inputs

example = make_example_views()
X, Y = example.X, example.Y

print("Reference dataset X")
display(X)
print("Comparison dataset Y")
display(Y)
print(f"d = {X.shape[1]}, n_X = {X.shape[0]}, n_Y = {Y.shape[0]}")

## 2. Missingness indicator matrices

Each entry of $M^X$ records whether one value in $X$ is missing:

$$
(M^X)_{ij}=
\begin{cases}
1, & x_{ij}\text{ is missing}\\
0, & x_{ij}\text{ is observed}
\end{cases}
$$

The matrix $M^Y$ is defined in the same way from $Y$. Therefore, $M^X$ has dimensions $n_X\times d$, and $M^Y$ has dimensions $n_Y\times d$.

In Python, we store $M^X$ as `M_X` and $M^Y$ as `M_Y`.

In [ ]:
#| label: missingness-indicator-matrices

M_X, M_Y = example.M_X, example.M_Y

print("M^X (stored as M_X)")
display(M_X)
print("M^Y (stored as M_Y)")
display(M_Y)
print(f"shape(M^X) = {M_X.shape}")
print(f"shape(M^Y) = {M_Y.shape}")

## 3. Missingness distribution for one variable

For variable $j$, the marginal missingness rates are

$$
r_j^X=\frac1{n_X}\sum_{i=1}^{n_X}(M^X)_{ij},
\qquad
r_j^Y=\frac1{n_Y}\sum_{i=1}^{n_Y}(M^Y)_{ij}
$$

The rate $r_j^X$ is the proportion of records where variable $j$ is missing in $X$. The rate $r_j^Y$ has the same meaning for $Y$.

Each rate defines a probability distribution with two groups: **observed** and **missing**. The two distributions are

$$
\mathbf p_j^X=(1-r_j^X,r_j^X),
\qquad
\mathbf p_j^Y=(1-r_j^Y,r_j^Y)
$$

In [ ]:
#| label: age-missingness-probabilities

r_X_age = float(M_X["age"].mean())
r_Y_age = float(M_Y["age"].mean())
p_X_age = np.array([1.0 - r_X_age, r_X_age])
p_Y_age = np.array([1.0 - r_Y_age, r_Y_age])

age_missingness_probabilities = pd.DataFrame(
    {
        "group": ["observed", "missing"],
        "p_X": p_X_age,
        "p_Y": p_Y_age,
    }
)
display(age_missingness_probabilities.round(4))
print(f"r_X_age = {r_X_age:.4f}")
print(f"r_Y_age = {r_Y_age:.4f}")

## 4. Hellinger distance for one variable

The marginal Hellinger distance for variable $j$ compares $\mathbf p_j^X$ with $\mathbf p_j^Y$:

$$
H_j^{\mathrm{marg}}=
\sqrt{\frac12\left[
(\sqrt{1-r_j^X}-\sqrt{1-r_j^Y})^2+
(\sqrt{r_j^X}-\sqrt{r_j^Y})^2
\right]}
$$

A value of $0$ means the variable has the same missingness rate in both datasets. A small value means the rates are similar. A large value means the rates are different. A value of $1$ means the variable is always observed in one dataset and always missing in the other.

In [ ]:
#| label: age-marginal-hellinger

H_age_marginal = hellinger_distance(p_X_age, p_Y_age)
print(f"H_age_marginal = {H_age_marginal:.6f}")

## 5. Average across variables with missingness

Let $\mathcal A$ contain the variables that are missing in at least one dataset:

$$
\mathcal A=\left\{j\in\{1,\ldots,d\}:
r_j^X>0\text{ or }r_j^Y>0\right\}
$$

The Marginal Missingness Hellinger Distance is the average of their column distances:

$$
\mathrm{Hel}_{\mathrm{miss}}^{\mathrm{marg}}(X,Y)=
\frac1{|\mathcal A|}\sum_{j\in\mathcal A}H_j^{\mathrm{marg}}
$$

Fully observed variables are excluded. This prevents unrelated complete variables from making the result smaller. If $\mathcal A$ is empty, the metric is defined as $0$.

In [ ]:
#| label: calculate-marginal-missingness

active_columns = [
    column
    for column in X.columns
    if X[column].isna().any() or Y[column].isna().any()
]
details = marginal_missingness_details(X, Y)
Hel_miss_marg = missingness_hellinger_distance(X, Y, mode="marginal")

print(f"Active columns A = {active_columns}")
display(details.round(4))
print(f"Marginal missingness Hellinger distance = {Hel_miss_marg:.6f}")
assert active_columns == details["column"].tolist()
assert np.isclose(Hel_miss_marg, details["distance"].mean())

In [ ]:
#| label: fig-marginal-missingness
#| fig-cap: "Missingness rates and marginal Hellinger distance for each active variable."

positions = np.arange(len(details))
width = 0.36
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

axes[0].bar(
    positions - width / 2,
    details["P_X(missing)"],
    width,
    label="X",
)
axes[0].bar(
    positions + width / 2,
    details["P_Y(missing)"],
    width,
    label="Y",
)
axes[0].set(
    title="Missingness rates",
    ylabel="Probability of missing",
    ylim=(0, 1),
    xticks=positions,
    xticklabels=details["column"],
)
axes[0].legend()

axes[1].bar(details["column"], details["distance"], color="#4c78a8")
axes[1].axhline(
    Hel_miss_marg,
    color="#e45756",
    linestyle="--",
    label=f"mean = {Hel_miss_marg:.3f}",
)
axes[1].set(
    title="Column distances",
    ylabel="Hellinger distance",
    ylim=(0, 1),
)
axes[1].legend()

for ax in axes:
    ax.tick_params(axis="x", rotation=35)
plt.show()

In [ ]:
#| label: metric-sanity-checks

Hel_miss_marg_reversed = missingness_hellinger_distance(Y, X, mode="marginal")
Hel_miss_marg_no_missingness = missingness_hellinger_distance(
    X.notna(),
    Y.notna(),
    mode="marginal",
)
Hel_miss_marg_with_complete_column = missingness_hellinger_distance(
    X.assign(complete_column=1),
    Y.assign(complete_column=2),
    mode="marginal",
)

print(f"Hel_miss_marg(Y, X) = {Hel_miss_marg_reversed:.6f}")
print(f"No missing values = {Hel_miss_marg_no_missingness:.6f}")
print(f"After adding a complete column = {Hel_miss_marg_with_complete_column:.6f}")
assert np.isclose(Hel_miss_marg, Hel_miss_marg_reversed)
assert np.isclose(Hel_miss_marg_no_missingness, 0.0)
assert np.isclose(Hel_miss_marg, Hel_miss_marg_with_complete_column)
assert 0.0 <= Hel_miss_marg <= 1.0

## 6. Interpretation

For the shared example, $\mathrm{Hel}_{\mathrm{miss}}^{\mathrm{marg}}(X,Y)\approx0.0305$. This is a small value. The active variables have similar missingness rates in $X$ and $Y$.

- A value of $0$ means every active variable has the same missingness rate in both datasets.
- A small value means the per-variable missingness rates are similar overall.
- A large value means the per-variable missingness rates are different overall.
- A value of $1$ means every averaged variable is always observed in one dataset and always missing in the other.
- The value is not the percentage of missing entries that differ.
- The metric is symmetric. Swapping $X$ and $Y$ does not change it.

A value of $0$ does not mean the same records contain missing values. It only means the column rates match. The next lesson compares complete row-wise missingness patterns.

## Takeaway

The Marginal Missingness Hellinger Distance follows four steps:

1. build $M^X$ and $M^Y$;
2. calculate the missingness rate of each variable;
3. compare the observed/missing distributions with Hellinger distance; and
4. average the distances over variables with missingness.

A smaller result means that $X$ and $Y$ have more similar per-variable missingness rates.